# Sesión 3, Parte B: pronóstico de inflación en un entorno rico en datos

**Curso: Proyecciones Macroeconómicas con Machine Learning y Deep Learning**

Replicaremos, en espíritu, el ejercicio estrella de la literatura ML-macro: **Medeiros, Vasconcelos, Veiga y Zilberman (2021, JBES)**, "Forecasting Inflation in a Data-Rich Environment". Target: la inflación mensual de EE.UU.; datos: FRED-MD, la base pública de McCracken y Ng. Al terminar podremos responder cinco preguntas:

1. ¿Cómo se monta un ejercicio "data-rich" estándar con FRED-MD y sus tcodes?
2. ¿Los ensambles de árboles le ganan al RW y al AR con 30 años de test?
3. ¿Dónde brilla Random Forest (el protagonista del paper) y dónde el shrinkage?
4. ¿Qué papel juegan los factores (componentes principales) como features?
5. ¿Qué mueve las predicciones del bosque?

> **La idea que guía la sesión**: el contrato de evaluación no cambia con el modelo. Y cuando la muestra es larga y hay no linealidades, los árboles por fin cobran lo que prometen.

In [ ]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from IPython.display import display
from sklearn.base import clone
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import Lasso, LinearRegression, Ridge
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit
from sklearn.pipeline import FeatureUnion, make_pipeline
from sklearn.preprocessing import FunctionTransformer, StandardScaler
from xgboost import XGBRegressor

# Localiza session3/ (utils.py + data/fredmd.csv).
candidatos = [Path.cwd(), Path.cwd() / "session3", *Path.cwd().parents]
SESSION_DIR = next(
    (
        ruta.resolve()
        for ruta in candidatos
        if (ruta / "utils.py").exists() and (ruta / "data" / "fredmd.csv").exists()
    ),
    None,
)
if SESSION_DIR is None:
    raise FileNotFoundError("No se encontró session3/data/fredmd.csv.")
if str(SESSION_DIR) not in sys.path:
    sys.path.insert(0, str(SESSION_DIR))

import utils as U

U.set_style()
OUTPUT = SESSION_DIR / "output"
OUTPUT.mkdir(exist_ok=True)

# ------------------------------------------------------------------
# Decisiones congeladas ANTES de mirar cualquier resultado del test.
# ------------------------------------------------------------------
HORIZON = 1
RANDOM_STATE = 42
N_FACTORS = 4                                  # factores PCA, como en el paper
WINDOW = 360                                   # rolling de 30 años, estilo Medeiros
TEST_FROM = pd.Period("1990-01", freq="M")     # primer target del test (como el paper)

SUBPERIODS_PRESET = {
    "1990-2000 (paper)": (pd.Period("1990-01", "M"), pd.Period("2000-12", "M")),
    "2001-2015 (paper)": (pd.Period("2001-01", "M"), pd.Period("2015-12", "M")),
    "2016-2026 (extensión)": (pd.Period("2016-01", "M"), None),
}
COVID_WINDOW = (pd.Period("2020-01", "M"), pd.Period("2021-12", "M"))   # diagnóstico declarado

MODEL_ORDER = ["RW", "Media", "AR(4)", "Ridge", "Lasso", "RandomForest", "XGBoost"]
BENCHMARK_MODEL = "RW"          # el benchmark titular del paper
PRIMARY_MODEL = "RandomForest"  # el protagonista de Medeiros; DM predefinido: RF vs RW

print(f"Sesión 3 | datos: {U.DATA}")
print(f"Test desde {TEST_FROM} | rolling window = {WINDOW} meses | {N_FACTORS} factores")

## 1. El contrato del caso (y qué replica del paper)

- **Target**: inflación mensual anualizada de EE.UU., $\pi_{t+1} = 1200\,\Delta\log(\text{CPIAUCSL}_{t+1})$.
- **Benchmarks**: Random Walk ($\widehat\pi_{t+1}=\pi_t$), media histórica y AR(4) directo.
- **Challengers**: Ridge, Lasso, Random Forest y XGBoost, todos con las mismas features.
- **Features estilo Medeiros**: 4 rezagos de cada serie de FRED-MD, 4 rezagos de la propia inflación y 4 factores (componentes principales) estimados dentro de cada ventana de entrenamiento.
- **Esquema**: ventana MÓVIL de 360 meses con reestimación mensual (como el paper), test 1990-2026 con los subperiodos del paper (1990-2000, 2001-2015) más nuestra extensión 2016-2026.
- **DM predefinido**: Random Forest vs RW.
- **Diagnóstico declarado**: la tabla sin los meses 2020-2021 (la regla de los dos extremos de siempre), por si la pandemia distorsiona.

**Qué NO replica**: sus 20+ modelos (UCSV, adaLASSO, redes profundas, model confidence sets), la rejilla completa de horizontes h = 1 a 12 y la inflación acumulada. Eso queda como práctica; el espíritu (misma base, mismo target, mismo tipo de evaluación) está completo.

In [ ]:
raw, tcodes, panel = U.load_fredmd()
pi = U.target_cpi_inflation(raw)

print(f"FRED-MD: {raw.shape[0]} meses x {raw.shape[1]} series ({raw.index.min()} a {raw.index.max()})")
print(f"Target: n = {pi.notna().sum()} | media = {pi.mean():.2f}% | DE = {pi.std():.2f} pp")

fig, ax = plt.subplots(figsize=(10, 4.0))
x = pi.index.to_timestamp()
ax.plot(x, pi, color=U.INK, lw=0.9, label="inflación mensual anualizada")
ax.plot(x, pi.rolling(12).mean(), color=U.ACCENT, lw=1.6, label="media móvil 12m")
ax.axvspan(TEST_FROM.to_timestamp(), x.max(), color=U.BLUE, alpha=0.08, label="test")
ax.axhline(0, color=U.BORDER, lw=0.8)
ax.set_ylabel("% anualizado")
ax.set_title("El target: inflación mensual de EE.UU. (CPIAUCSL)", loc="left")
ax.legend(fontsize=9, loc="upper right")
U.save_fig(fig, "D01_inflacion_target")
plt.show()

## 2. FRED-MD: la base mensual de McCracken y Ng

FRED-MD trae 126 series mensuales de EE.UU. desde 1959, cada una con su código de transformación (aquí la fila especial es una sola: `Transform:`). La lección de estacionariedad de la Sesión 1 viene resuelta de fábrica; nuestro target usa $\Delta\log$ del índice de precios, la definición del paper.

Limpieza declarada: muestra desde 1962, fuera las series con más de 5% de huecos, y filas completas.

**Predicción antes de ejecutar:** ¿cuántas de las 126 series sobreviven?

In [ ]:
print("Distribución de tcodes:", tcodes.value_counts().sort_index().to_dict())

sub = panel.loc["1962-01":]
frac_nan = sub.isna().mean()
KEEP = frac_nan[frac_nan <= 0.05].index.tolist()
clean = sub[KEEP].dropna()
print(f"Series: {panel.shape[1]} totales -> {len(KEEP)} sobreviven (<= 5% de huecos)")
print(f"Panel limpio: {clean.shape[0]} meses ({clean.index.min()} a {clean.index.max()})")

fig, ax = plt.subplots(figsize=(10, 3.4))
disponibles = panel.notna().sum(axis=1)
ax.plot(disponibles.index.to_timestamp(), disponibles, color=U.INK, lw=1.2)
ax.axhline(panel.shape[1], color=U.BORDER, lw=0.8, ls="dashed")
ax.axvline(pd.Period("1962-01", "M").to_timestamp(), color=U.ACCENT, lw=1.0, ls="dashed")
ax.set_ylabel("series con dato")
ax.set_title("Disponibilidad en FRED-MD: las series no nacen todas el mismo día", loc="left")
U.save_fig(fig, "D02_fredmd_disponibilidad")
plt.show()

## 3. Features estilo Medeiros: rezagos + factores

Tres bloques, congelados:

1. **4 rezagos de cada serie** del panel transformado (la dinámica reciente de todo);
2. **4 rezagos de la propia inflación** (la parte AR del problema);
3. **4 factores** (componentes principales del panel estandarizado), estimados \emph{dentro} de cada ventana de entrenamiento vía `FeatureUnion` + `PCA` en el `Pipeline`: cero leakage, la lección de la Sesión 2 aplicada a la reducción de dimensión.

El resultado: $p$ cerca de 500 features con $n = 360$ meses por ventana. Otra "fat regression". La diferencia con los ejercicios trimestrales de macro es el tamaño: aquí cada ventana tiene 360 meses y el test 400+ observaciones.

In [ ]:
bloques = []
for lag in range(1, 5):
    bloques.append(clean.shift(lag - 1).add_suffix(f"_l{lag}"))   # l1 = valor en t
ar_block = pd.DataFrame({f"pi_l{k}": pi.shift(k - 1) for k in range(1, 5)})
AR_FEATURES = list(ar_block.columns)

df = pd.concat(bloques + [ar_block], axis=1)
df["y_next"] = pi.shift(-HORIZON)
df["target_m"] = df.index + HORIZON
df = df.dropna()
ALL_FEATURES = [c for c in df.columns if c not in ("y_next", "target_m")]

print(f"Panel de modelado: n = {len(df)} orígenes ({df.index.min()} a {df.index.max()})")
print(f"Features: p = {len(ALL_FEATURES)} + {N_FACTORS} factores estimados por ventana")


def augmentador():
    """Features originales + factores PCA estimados con el train de cada ajuste."""
    return FeatureUnion([
        ("originales", FunctionTransformer(feature_names_out="one-to-one")),
        ("factores", make_pipeline(StandardScaler(), PCA(n_components=N_FACTORS,
                                                         random_state=RANDOM_STATE))),
    ])

## 4. Split temporal: la ventana móvil del paper

Medeiros et al. reestiman cada mes con una ventana móvil: la elección clásica cuando se sospecha que las relaciones cambian (recordar la disyuntiva expansiva vs móvil de la Sesión 2). La seguimos: **360 meses de train, reestimación mensual**.

- **Development** (tuning): targets hasta 1989-12, con cinco folds temporales de 24 meses.
- **Test**: targets desde 1990-01, bloqueado. Son 36 años y 400+ orígenes: uno de los tests más largos que veremos en el curso.
- Subperiodos: los del paper (1990-2000, 2001-2015) más la extensión 2016-2026. El diagnóstico "sin 2020-2021" queda declarado por si la pandemia distorsiona; con inflación mensual el golpe es acotado, así que el titular incluye todo.

In [ ]:
development = df[df["target_m"] < TEST_FROM]
test = df[df["target_m"] >= TEST_FROM]
assert development["target_m"].max() < test["target_m"].min()

TSCV = TimeSeriesSplit(n_splits=5, test_size=24)
CV_SPLITS = list(TSCV.split(development))


def covid_pair(origin_index, horizon=HORIZON, window=COVID_WINDOW):
    """True si el origen o su target caen en la ventana 2020-2021 declarada."""
    origin = pd.PeriodIndex(origin_index)
    target = origin + horizon
    lo, hi = window
    touch = lambda p: (p >= lo) & (p <= hi)
    return touch(origin) | touch(target)


print(f"Development: {len(development)} targets ({development['target_m'].min()} a {development['target_m'].max()})")
print(f"Test: {len(test)} targets ({test['target_m'].min()} a {test['target_m'].max()})")
print(f"Pares que tocan 2020-2021 (solo diagnóstico): {int(covid_pair(test.index).sum())}")

fig, ax = plt.subplots(figsize=(10, 3.8))
serie = pi.loc["1962-01":]
x = serie.index.to_timestamp()
ax.plot(x, serie.rolling(12).mean(), color=U.INK, lw=1.3, label="inflación (media móvil 12m)")
ax.axvspan(x.min(), pd.Period("1989-12", "M").to_timestamp(how="end"),
           color=U.TINT, alpha=0.5, label="development (tuning)")
ax.axvspan(TEST_FROM.to_timestamp(), x.max(), color=U.BLUE, alpha=0.10, label="test 1990+")
ax.axhline(0, color=U.BORDER, lw=0.8)
ax.set_ylabel("% anualizado")
ax.set_title("36 años de test; el train es una ventana móvil de 30 años", loc="left")
ax.legend(fontsize=9, loc="upper right")
U.save_fig(fig, "D03_split_ventana")
plt.show()

## 5. Tuning con folds temporales dentro de development

Grids chicos y declarados, con el `augmentador` (rezagos + factores) dentro de cada pipeline: los factores se reestiman en cada fold, sin espiar. Los frenos de siempre: profundidad mínima y learning rate bajo en XGBoost; hojas grandes y pocas features por corte en el bosque.

**Predicción antes de ejecutar:** con muestra mensual (el triple que la trimestral), ¿los árboles ya compiten con los lineales penalizados en validación?

In [ ]:
SEARCH_SPACES = {
    "Ridge": (
        make_pipeline(augmentador(), StandardScaler(), Ridge(solver="svd")),
        {"ridge__alpha": np.logspace(-2, 6, 25)},
    ),
    "Lasso": (
        make_pipeline(augmentador(), StandardScaler(), Lasso(max_iter=50_000, tol=1e-3)),
        {"lasso__alpha": np.logspace(-3, 1, 20)},
    ),
    "RandomForest": (
        make_pipeline(augmentador(), RandomForestRegressor(
            n_estimators=500, random_state=RANDOM_STATE, n_jobs=-1)),
        {"randomforestregressor__max_depth": [None, 6],
         "randomforestregressor__min_samples_leaf": [5, 10],
         "randomforestregressor__max_features": [0.33, 1.0]},
    ),
    "XGBoost": (
        make_pipeline(augmentador(), XGBRegressor(
            subsample=0.8, colsample_bytree=0.7, reg_lambda=1.0,
            random_state=RANDOM_STATE, n_jobs=4)),
        {"xgbregressor__learning_rate": [0.03, 0.1],
         "xgbregressor__max_depth": [2, 3],
         "xgbregressor__n_estimators": [200, 500]},
    ),
}

TUNED = {}
BEST_PARAMS = {}
tuning_rows = []
for name, (estimator, grid) in SEARCH_SPACES.items():
    search = GridSearchCV(estimator, grid, cv=CV_SPLITS,
                          scoring="neg_root_mean_squared_error")
    search.fit(development[ALL_FEATURES], development["y_next"])
    TUNED[name] = clone(search.best_estimator_)
    BEST_PARAMS[name] = {k.split("__")[-1]: (v.item() if isinstance(v, np.generic) else v)
                         for k, v in search.best_params_.items()}
    tuning_rows.append({"modelo": name, "RMSE_validation": -search.best_score_})
    print(f"{name:13s} | RMSE validation = {-search.best_score_:.3f} | {BEST_PARAMS[name]}")

ar4 = LinearRegression()
ar_scores, media_scores, rw_scores = [], [], []
for idx_tr, idx_va in CV_SPLITS:
    tr, va = development.iloc[idx_tr], development.iloc[idx_va]
    ar4.fit(tr[AR_FEATURES], tr["y_next"])
    ar_scores.append(U.rmse(va["y_next"] - ar4.predict(va[AR_FEATURES])))
    media_scores.append(U.rmse(va["y_next"] - tr["y_next"].mean()))
    rw_scores.append(U.rmse(va["y_next"] - va["pi_l1"]))
tuning_rows += [
    {"modelo": "AR(4)", "RMSE_validation": float(np.mean(ar_scores))},
    {"modelo": "Media", "RMSE_validation": float(np.mean(media_scores))},
    {"modelo": "RW", "RMSE_validation": float(np.mean(rw_scores))},
]

tuning_table = (pd.DataFrame(tuning_rows).set_index("modelo")
                .loc[MODEL_ORDER].round(3))
display(tuning_table)

fig, ax = plt.subplots(figsize=(8.5, 3.8))
orden = tuning_table["RMSE_validation"].sort_values(ascending=False)
ax.barh(orden.index, orden,
        color=[U.ACCENT if m == orden.idxmin() else U.TINT for m in orden.index],
        edgecolor=U.BORDER)
ax.set_xlabel("RMSE de validación (pp anualizados)")
ax.set_title("Validation elige la complejidad; el test aún no juega", loc="left")
U.save_fig(fig, "D04_tuning_validacion")
plt.show()

## 6. [Opcional] Tuning bayesiano con BayesSearchCV

Mismo contrato, otro buscador: `BayesSearchCV` (scikit-optimize) aprende de las evaluaciones anteriores. Activa `RUN_BAYES = True` para correrlo (tarda varios minutos).

In [ ]:
RUN_BAYES = False

if RUN_BAYES:
    from skopt import BayesSearchCV
    from skopt.space import Integer, Real

    bayes_rf = BayesSearchCV(
        make_pipeline(augmentador(), RandomForestRegressor(
            n_estimators=500, random_state=RANDOM_STATE, n_jobs=-1)),
        {"randomforestregressor__max_depth": Integer(3, 20),
         "randomforestregressor__min_samples_leaf": Integer(2, 20),
         "randomforestregressor__max_features": Real(0.1, 1.0)},
        n_iter=25, cv=CV_SPLITS, scoring="neg_root_mean_squared_error",
        random_state=RANDOM_STATE,
    )
    bayes_rf.fit(development[ALL_FEATURES], development["y_next"])
    print(f"BayesSearchCV RF | RMSE validation = {-bayes_rf.best_score_:.3f}")
    print(f"  mejores hiperparámetros: {dict(bayes_rf.best_params_)}")
    print(f"  (el grid chico logró {tuning_table.loc['RandomForest', 'RMSE_validation']:.3f})")
else:
    print("Sección opcional: pon RUN_BAYES = True para explorar con scikit-optimize.")

## 7. Backtest con ventana móvil y cache

El loop de siempre, con la ventana móvil del paper: en cada origen se entrena con los últimos 360 meses ya observados y se pronostica el mes siguiente. Hiperparámetros congelados; los factores se reestiman dentro de cada ventana (van en el pipeline).

La corrida completa (400+ orígenes, 8 modelos) tarda varios minutos y se guarda en `output/backtest_fredmd.parquet`: las siguientes ejecuciones cargan el cache en segundos.

In [ ]:
from tqdm.auto import tqdm

CACHE = OUTPUT / "backtest_fredmd.parquet"
FORCE_RERUN = False

MODELS = {"RW": None, "Media": None, "AR(4)": LinearRegression(), **TUNED}


def walk_forward(df, test_origins):
    """Backtest con ventana móvil de WINDOW meses y decisiones congeladas."""
    rows = []
    for origin in tqdm(test_origins, desc="Backtest móvil"):
        past = df[df["target_m"] <= origin]
        train = past.tail(WINDOW)
        current = df.loc[[origin]]

        preds = {"RW": float(current["pi_l1"].iloc[0]),
                 "Media": float(train["y_next"].mean())}
        ar_m = clone(MODELS["AR(4)"]).fit(train[AR_FEATURES], train["y_next"])
        preds["AR(4)"] = float(ar_m.predict(current[AR_FEATURES])[0])
        for name in MODEL_ORDER[3:]:
            model = clone(MODELS[name])
            model.fit(train[ALL_FEATURES], train["y_next"])
            preds[name] = float(model.predict(current[ALL_FEATURES])[0])

        y_true = float(current["y_next"].iloc[0])
        for name, y_hat in preds.items():
            rows.append({"origin": str(origin),
                         "target_m": str(current["target_m"].iloc[0]),
                         "model": name, "y_true": y_true, "y_hat": y_hat})
    out = pd.DataFrame(rows)
    out["error"] = out["y_true"] - out["y_hat"]
    return out


if CACHE.exists() and not FORCE_RERUN:
    backtest = pd.read_parquet(CACHE)
    print(f"Cache cargado: {CACHE.name} ({len(backtest)} pronósticos)")
else:
    backtest = walk_forward(df, test.index)
    backtest.to_parquet(CACHE, index=False)
    print(f"Backtest ejecutado y guardado en {CACHE.name}")

backtest["target_m"] = pd.PeriodIndex(backtest["target_m"], freq="M")
backtest["origin"] = pd.PeriodIndex(backtest["origin"], freq="M")
backtest["covid"] = covid_pair(backtest["origin"])
print(f"{backtest['origin'].nunique()} orígenes x {backtest['model'].nunique()} modelos = {len(backtest)}")

## 8. Resultados: los subperiodos del paper y la extensión

`relativo` divide cada RMSE entre el del RW, la métrica titular de Medeiros: menor que 1 es ganarle al benchmark. El contraste predefinido es Random Forest vs RW con Diebold-Mariano; reportamos también la comparación con el AR(4), el benchmark exigente.

In [ ]:
def metric_table(sample):
    rows = []
    rmse_rw = U.rmse(sample.loc[sample["model"] == "RW", "error"])
    rmse_ar = U.rmse(sample.loc[sample["model"] == "AR(4)", "error"])
    for name in MODEL_ORDER:
        e = sample.loc[sample["model"] == name, "error"]
        rows.append({"modelo": name, "RMSE": U.rmse(e),
                     "rel_RW": U.rmse(e) / rmse_rw, "rel_AR4": U.rmse(e) / rmse_ar})
    return pd.DataFrame(rows).set_index("modelo")

print("=== Test completo 1990-2026 ===")
display(metric_table(backtest).round(3))

sub_rows = []
for periodo, (ini, fin) in SUBPERIODS_PRESET.items():
    fin = backtest["target_m"].max() if fin is None else fin
    s = backtest[backtest["target_m"].between(ini, fin)]
    for name in MODEL_ORDER:
        e = s.loc[s["model"] == name, "error"]
        rw = U.rmse(s.loc[s["model"] == "RW", "error"])
        sub_rows.append({"periodo": periodo, "modelo": name, "rel_RW": U.rmse(e) / rw})
sub_table = pd.DataFrame(sub_rows).pivot(index="modelo", columns="periodo", values="rel_RW")
print("=== RMSE relativo al RW por subperiodo ===")
display(sub_table.loc[MODEL_ORDER].round(3))

print("=== Diagnóstico declarado: sin 2020-2021 ===")
display(metric_table(backtest[~backtest["covid"]]).round(3)[["RMSE", "rel_RW"]])

errores = backtest.pivot(index="target_m", columns="model", values="error")
dm_stat, dm_p = U.dm_test(errores[PRIMARY_MODEL], errores[BENCHMARK_MODEL], h=HORIZON)
print(f"DM {PRIMARY_MODEL} vs {BENCHMARK_MODEL}: estadístico = {dm_stat:.2f}, p-valor = {dm_p:.4f}")
dm2, p2 = U.dm_test(errores[PRIMARY_MODEL], errores["AR(4)"], h=HORIZON)
print(f"DM {PRIMARY_MODEL} vs AR(4):     estadístico = {dm2:.2f}, p-valor = {p2:.4f}")

In [ ]:
actuals = backtest.drop_duplicates("target_m").set_index("target_m")["y_true"]
pred = backtest.pivot(index="target_m", columns="model", values="y_hat")

fig, ax = plt.subplots(figsize=(10, 4.0))
x = actuals.index.to_timestamp()
ax.plot(x, actuals.rolling(3).mean(), color=U.INK, lw=1.1, label="inflación (media móvil 3m)")
ax.plot(x, pred[PRIMARY_MODEL].rolling(3).mean(), color=U.ACCENT, lw=1.2, ls="dashed",
        label=f"{PRIMARY_MODEL} (media móvil 3m)")
ax.axhline(0, color=U.BORDER, lw=0.8)
ax.set_ylabel("% anualizado")
ax.set_title("Test 1990-2026: la inflación mensual es ruidosa; el bosque sigue su centro", loc="left")
ax.legend(fontsize=9, loc="upper right")
U.save_fig(fig, "D05_pronosticos_test")
plt.show()

gan = (errores["RW"] ** 2 - errores[PRIMARY_MODEL] ** 2).dropna().cumsum()
fig, ax = plt.subplots(figsize=(10, 3.2))
ax.plot(gan.index.to_timestamp(), gan, color=U.BLUE, lw=1.6)
ax.axhline(0, color=U.MUTED, lw=0.8)
ax.set_ylabel("ganancia acumulada")
ax.set_title(f"Pérdida cuadrática acumulada: positivo favorece a {PRIMARY_MODEL} sobre el RW", loc="left")
U.save_fig(fig, "D06_ganancia_acumulada")
plt.show()

## 9. Importancias del bosque: gain vs permutación

Dos lupas sobre el Random Forest final (ajustado con la última ventana de development):

- **Gain** (impureza): cuánto redujo el error cada feature en los cortes. Gratis, pero reparte el crédito arbitrariamente entre features correlacionadas.
- **Permutación**: cuánto empeora el RMSE al barajar cada feature en un bloque temporal de validación. Más honesta, más cara.

Con ~500 features correlacionadas las listas difieren, y esa discrepancia es la lección. Ambas llegan *después* del modelo: en la Sesión 4 veremos la alternativa, una red cuya descomposición aditiva viene de su propia arquitectura.

In [ ]:
from sklearn.inspection import permutation_importance

corte = int(len(development) * 0.8)
dev_train, dev_val = development.iloc[:corte], development.iloc[corte:]

rf_final = clone(TUNED[PRIMARY_MODEL])
rf_final.fit(dev_train[ALL_FEATURES], dev_train["y_next"])

nombres = ALL_FEATURES + [f"factor_{k}" for k in range(1, N_FACTORS + 1)]
rf_core = rf_final.named_steps["randomforestregressor"]
gain = pd.Series(rf_core.feature_importances_, index=nombres)
top_gain = gain.nlargest(15)

perm = permutation_importance(
    rf_final, dev_val[ALL_FEATURES], dev_val["y_next"],
    n_repeats=10, random_state=RANDOM_STATE, scoring="neg_root_mean_squared_error",
)
perm_imp = pd.Series(perm.importances_mean, index=ALL_FEATURES)
top_perm = perm_imp.nlargest(15)

fig, axes = plt.subplots(1, 2, figsize=(11, 4.6))
axes[0].barh(top_gain.index[::-1], top_gain[::-1], color=U.ACCENT, edgecolor=U.BORDER)
axes[0].set_title("Gain (Random Forest)", loc="left", fontsize=10)
axes[1].barh(top_perm.index[::-1], top_perm[::-1], color=U.BLUE, edgecolor=U.BORDER)
axes[1].set_title("Permutación (bloque temporal de validación)", loc="left", fontsize=10)
for ax in axes:
    ax.tick_params(axis="y", labelsize=7)
fig.suptitle("Dos lupas, dos rankings: la correlación reparte el crédito",
             x=0.02, ha="left", fontweight="bold")
U.save_fig(fig, "D07_importancias_gain_perm")
plt.show()

comunes = set(top_gain.index) & set(top_perm.index)
print(f"Features en ambos top-15: {len(comunes)} de 15")
print("Nota: la permutación se calcula sobre las features originales; los factores")
print("viven dentro del pipeline y absorben parte del crédito en el gain.")

## 10. Síntesis

**Qué replica del espíritu de Medeiros et al. (2021):** la base (FRED-MD con tcodes), el target (inflación de EE.UU.), las features (rezagos + factores), la ventana móvil con reestimación mensual, los subperiodos del paper y su métrica titular (RMSE relativo al RW). Y, sobre todo, el mensaje: **con muestra larga y no linealidades, el ML le gana a los benchmarks**, con Random Forest brillando en los 90 y el shrinkage firme después.

**Qué NO replica:** su batería completa de modelos y horizontes, los model confidence sets, y los vintages en tiempo real (usamos un panel revisado, la convención de la literatura FRED).

**Las lecciones que viajan:**

1. El contrato no cambia con el modelo: mismos folds, mismos benchmarks, mismas reglas para todos.
2. La muestra manda: 360 meses por ventana y 429 orígenes de test son lo que permite que la no linealidad se pague. Con muestras cortas, el mismo pipeline suele perder contra un benchmark simple.
3. Los factores son features honestas si se estiman dentro de la ventana (Pipeline), nunca con la muestra completa.
4. Gain y permutación siguen contando historias distintas; ninguna es causal.

### Para practicar

- Replica el otro titular del paper: horizontes directos h = 3, 6 y 12 y la inflación acumulada a 12 meses. ¿A qué horizonte crece la ventaja del bosque?
- Cambia el target a otra medida de precios del panel (PCE: `PCEPI`). ¿Sobrevive el ranking?
- Cambia la ventana móvil por una expansiva: ¿quién gana? (El paper eligió móvil; ahora sabes discutirle.)
- Construye el ensamble por mediana de los cinco challengers y compáralo con el mejor individual.
- Activa `RUN_BAYES = True` y compara el tuning bayesiano del bosque con el grid chico.
